# Activation Geometry

Clone the repo into Kaggle, measure last-token residual activations on the discovery split, and compute the first direction artifacts.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("FRS_REPO_URL", "").strip()
REPO_DIR = Path("/kaggle/working/false-refusal-suppression")

if not REPO_URL:
    raise RuntimeError("Set FRS_REPO_URL in the Kaggle environment before running this notebook.")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[train,dev]"], check=True)

os.chdir(REPO_DIR)
src_path = REPO_DIR / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(REPO_DIR)

In [ ]:
from pathlib import Path

MODEL_ID = "Qwen/Qwen3-4B"
SPLIT_DIR = Path("data/processed/splits/sample_small")
ACTIVATION_ARTIFACT = Path("artifacts/activations/qwen3_sample_discovery.json")
DIRECTION_ARTIFACT = Path("artifacts/directions/qwen3_sample_borderline_vs_easy.json")

print(MODEL_ID)
print(ACTIVATION_ARTIFACT)
print(DIRECTION_ARTIFACT)

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/make_splits.py",
        "--input",
        "data/processed/prompts/sample_prompts.jsonl",
        "--output-dir",
        str(SPLIT_DIR),
        "--config",
        "configs/data/prompt_sets_small.yaml",
        "--seed",
        "7",
    ],
    check=True,
 )

subprocess.run(
    [
        sys.executable,
        "scripts/measure_activations.py",
        "--model-id",
        MODEL_ID,
        "--split-path",
        str(SPLIT_DIR / "discovery.jsonl"),
        "--output",
        str(ACTIVATION_ARTIFACT),
        "--capture-default-modules",
        "--max-module-captures",
        "12",
    ],
    check=True,
 )

subprocess.run(
    [
        sys.executable,
        "scripts/compute_directions.py",
        "--activations",
        str(ACTIVATION_ARTIFACT),
        "--source-group-a",
        "benign_borderline",
        "--source-group-b",
        "benign_easy",
        "--output",
        str(DIRECTION_ARTIFACT),
    ],
    check=True,
 )

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

with open(ACTIVATION_ARTIFACT, "r", encoding="utf-8") as handle:
    activation_artifact = json.load(handle)

with open(DIRECTION_ARTIFACT, "r", encoding="utf-8") as handle:
    direction_artifact = json.load(handle)

ranked_layers = pd.DataFrame(direction_artifact["ranked_layers"])
display(pd.DataFrame([activation_artifact["summary"]]))
display(ranked_layers.head(12))

if not ranked_layers.empty:
    ax = ranked_layers.head(12).plot.bar(
        x="name",
        y="score",
        figsize=(10, 4),
        legend=False,
        title="Top separability scores by layer",
    )
    ax.set_ylabel("separability score")
    plt.xticks(rotation=45, ha="right")
    plt.show()